In [2]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

print("Loading 'pv_shading_dataset.csv'...")
try:
    df = pd.read_csv("pv_shading_dataset.csv")
except FileNotFoundError:
    print("\n[ERROR]: Could not find 'pv_shading_dataset.csv'!")
    print("Please make sure you ran the Phase 1 Data Generation script first.")
    exit()

print(f"Dataset successfully loaded: {df.shape[0]} rows of observations found.")

X = df[['Output_Voltage', 'Output_Power']]

mod_cols = [f"Mod_{i}_G" for i in range(9) if f"Mod_{i}_G" in df.columns]

# Split into 80% Training data (for learning) and 20% Testing data (for validation)
df.rename(columns={'Optimal_Config_Label': 'Optimal_Config'}, inplace=True)
df['Shaded_Module'] = df[mod_cols].idxmin(axis=1).str.extract(r'(\d+)').astype(int)
y = df[['Shaded_Module', 'Optimal_Config']]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"  Allocation for Training: {X_train.shape[0]} rows")
print(f"  Allocation for Evaluation: {X_test.shape[0]} rows")

print("\nTraining the Multi-Output Machine Learning Model...")
print("The algorithm is mapping Voltage/Power curves to structural faults...")

model = RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42)
model.fit(X_train, y_train)

print("Training Complete!")

print("\n=================== SYSTEM EVALUATION ===================")
y_pred = model.predict(X_test)

acc_module = accuracy_score(y_test['Shaded_Module'], y_pred[:, 0])
acc_config = accuracy_score(y_test['Optimal_Config'], y_pred[:, 1])

print(f" -> Shaded Module Prediction Accuracy: {acc_module * 100:.2f}%")
print(f" -> Optimal Configuration Prediction Accuracy: {acc_config * 100:.2f}%")

exact_match = np.all(y_test.values == y_pred, axis=1)

Loading 'pv_shading_dataset.csv'...
Dataset successfully loaded: 4000 rows of observations found.
  Allocation for Training: 3200 rows
  Allocation for Evaluation: 800 rows

Training the Multi-Output Machine Learning Model...
The algorithm is mapping Voltage/Power curves to structural faults...
Training Complete!

=================== SYSTEM EVALUATION ===================
 -> Shaded Module Prediction Accuracy: 17.62%
 -> Optimal Configuration Prediction Accuracy: 60.50%
